# Etapa 4 — Segmentação dos pagantes ativos

**Pergunta:** quais grupos comportamentais acionáveis existem entre os 14.753 pagantes ativos em 30/09/2026?

**Regra central:** os clusters são formados sem churn, status de saída, pesquisa de saída, campanha ou qualquer fato posterior ao corte. Demografia e renovação automática são reservadas apenas para auditoria e perfil posterior.

## 1. Fontes e linhagem

O processamento consome exclusivamente Parquets da camada Prata: assinaturas, usuários, onboarding, atividades, uso mensal, pagamentos, amizades, feed e suporte. O script oficial é `src/segmentacao_clientes.py`; os produtos individuais permanecem na camada Ouro e apenas agregados seguros são exportados para `saidas/tabelas/`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

ROOT = Path('..')
TABELAS = ROOT / 'saidas' / 'tabelas'
OURO = ROOT / 'dados' / 'processados' / 'ouro'
metadados = json.loads((TABELAS / 'metadados_segmentacao.json').read_text(encoding='utf-8'))
metadados

## 2. Definition of Done e controles

- exatamente 14.753 usuários únicos;
- uma linha por pagante ativo;
- nenhuma feature de saída ou pós-tratamento;
- comparação de 3 e 4 clusters;
- tamanho mínimo de 5%;
- estabilidade por reamostragem;
- perfil estatístico e validação nos cinco critérios de negócio.

In [ ]:
features = pd.read_parquet(OURO / 'features_segmentacao_ativos_set2026.parquet')
assert len(features) == 14_753
assert features['user_id'].is_unique
proibidas = {'churn','motivo_fim','data_fim','motivo_pesquisa_saida','abriu_push','abriu_email','clicou'}
assert not proibidas.intersection(metadados['features_cluster'])
features.shape

## 3. Escolha de k

K-Means foi avaliado com inicializações múltiplas. A solução oficial é escolhida apenas entre `k=3` e `k=4`; `k=5` funciona como diagnóstico. Silhouette e Davies–Bouldin medem separação, enquanto o ARI médio mede estabilidade frente a reamostragens.

In [ ]:
metricas = pd.read_csv(TABELAS / 'metricas_clusterizacao.csv')
display(metricas)
display(Image(filename=str(ROOT / 'saidas' / 'segmentacao_validacao_k.png')))

## 4. Perfis encontrados

A solução de três grupos separa um conjunto sem hábito recente e dois conjuntos ativos, diferenciados principalmente pelo modo de treino: atividades ao ar livre versus vídeos guiados. Os nomes descrevem centroides agregados; não são rótulos psicológicos nem inferências sobre indivíduos.

In [ ]:
perfil = pd.read_csv(TABELAS / 'perfil_segmentos.csv')
display(perfil)
display(Image(filename=str(ROOT / 'saidas' / 'segmentacao_heatmap_perfis.png')))
display(Image(filename=str(ROOT / 'saidas' / 'segmentacao_tamanho_receita.png')))

## 5. Separação e limitação

A estabilidade elevada não elimina a sobreposição entre os dois grupos ativos. A projeção PCA e o silhouette baixo mostram que a base não contém fronteiras naturais fortes em todas as dimensões. A segmentação deve ser usada como estrutura descritiva e acionável, não como verdade ontológica nem como modelo de churn.

In [ ]:
display(Image(filename=str(ROOT / 'saidas' / 'segmentacao_pca.png')))
pd.read_csv(TABELAS / 'criterios_segmentacao.csv')

## 6. Próximo uso

Na Etapa 5, os três perfis serão convertidos em cartões de persona baseados em dados, hipóteses de drivers e priorização preliminar. A taxa de renovação desligada pode ajudar a interpretar os clusters depois de formados, mas não participou da distância do algoritmo.